In [ ]:
import os, urllib.request, zipfile, io, time
import pandas as pd
from pathlib import Path

RAW = "data/raw"
os.makedirs(RAW, exist_ok=True)
print(f"Raw data directory: {os.path.abspath(RAW)}")


## 1. APL — Accessibilité Potentielle Localisée (DREES)

In [ ]:
APL_PATH = f"{RAW}/apl.csv"
APL_URL = "https://www.data.gouv.fr/fr/datasets/r/0bb9c29e-3fc4-4a45-a62b-3d8b0f0e5ead"
APL_MIN_SIZE = 1_000_000  # 1 MB minimum

try:
    if os.path.exists(APL_PATH) and os.path.getsize(APL_PATH) > APL_MIN_SIZE:
        print(f"APL: Already downloaded ({os.path.getsize(APL_PATH):,} bytes)")
    else:
        print("Downloading APL...")
        urllib.request.urlretrieve(APL_URL, APL_PATH)
        print(f"APL: Downloaded ({os.path.getsize(APL_PATH):,} bytes)")

    # Validate
    apl_df = pd.read_csv(APL_PATH, nrows=5, sep=None, engine='python')
    print(f"APL columns: {list(apl_df.columns[:5])}")

    # Row count
    apl_rows = sum(1 for _ in open(APL_PATH)) - 1
    print(f"APL rows: {apl_rows:,}")
    assert apl_rows > 30_000, f"APL too few rows: {apl_rows}"
    cols_lower = [c.lower() for c in apl_df.columns]
    assert any("commune" in c or "apl" in c or "com" in c for c in cols_lower), f"APL missing commune/apl col: {list(apl_df.columns)}"
    print("APL: OK")
except Exception as e:
    print(f"APL ERROR: {e}")
    import traceback; traceback.print_exc()


## 2. RPPS — Annuaire Santé (Professionnels de Santé)

In [ ]:
RPPS_PATH = f"{RAW}/rpps.csv"
RPPS_URL = "https://www.data.gouv.fr/fr/datasets/r/e85b2bfe-6571-4616-9a1e-77ded1a82c8e"
RPPS_MIN_SIZE = 100_000_000  # 100 MB minimum

try:
    if os.path.exists(RPPS_PATH) and os.path.getsize(RPPS_PATH) > RPPS_MIN_SIZE:
        print(f"RPPS: Already downloaded ({os.path.getsize(RPPS_PATH):,} bytes)")
    else:
        print("Downloading RPPS (~761 MB, may take several minutes)...")
        def reporthook(count, block_size, total_size):
            if total_size > 0 and count % 500 == 0:
                pct = count * block_size / total_size * 100
                print(f"  {pct:.1f}%", end="\r")
        urllib.request.urlretrieve(RPPS_URL, RPPS_PATH, reporthook=reporthook)
        print(f"\nRPPS: Downloaded ({os.path.getsize(RPPS_PATH):,} bytes)")

    # Validate file size
    rpps_size = os.path.getsize(RPPS_PATH)
    assert rpps_size > RPPS_MIN_SIZE, f"RPPS file too small: {rpps_size:,} bytes (expected >100MB)"

    # Try reading with UTF-8, fallback to Latin-1
    try:
        rpps_sample = pd.read_csv(RPPS_PATH, nrows=5, encoding='utf-8', sep='|', low_memory=False)
        encoding_used = 'utf-8'
    except (UnicodeDecodeError, Exception):
        rpps_sample = pd.read_csv(RPPS_PATH, nrows=5, encoding='latin-1', sep='|', low_memory=False)
        encoding_used = 'latin-1'
    print(f"RPPS encoding: {encoding_used}")
    print(f"RPPS columns: {list(rpps_sample.columns[:5])}")

    # Count rows (fast)
    with open(RPPS_PATH, encoding=encoding_used, errors='replace') as f:
        rpps_rows = sum(1 for _ in f) - 1
    print(f"RPPS rows: {rpps_rows:,}")
    assert rpps_rows > 1_000_000, f"RPPS too few rows: {rpps_rows:,} (expected >1M)"
    print("RPPS: OK")
except Exception as e:
    print(f"RPPS ERROR: {e}")
    import traceback; traceback.print_exc()


## 3. FINESS — Fichier des Établissements de Santé

In [ ]:
FINESS_PATH = f"{RAW}/finess.csv"
FINESS_URL = "https://www.data.gouv.fr/fr/datasets/r/2ce43ade-8d2c-4d1d-81da-ca06c82abc68"
FINESS_MIN_SIZE = 5_000_000  # 5 MB minimum

try:
    if os.path.exists(FINESS_PATH) and os.path.getsize(FINESS_PATH) > FINESS_MIN_SIZE:
        print(f"FINESS: Already downloaded ({os.path.getsize(FINESS_PATH):,} bytes)")
    else:
        print("Downloading FINESS...")
        urllib.request.urlretrieve(FINESS_URL, FINESS_PATH)
        print(f"FINESS: Downloaded ({os.path.getsize(FINESS_PATH):,} bytes)")

    # Validate
    try:
        finess_df = pd.read_csv(FINESS_PATH, nrows=5, sep=';', encoding='utf-8', low_memory=False)
    except Exception:
        finess_df = pd.read_csv(FINESS_PATH, nrows=5, sep=';', encoding='latin-1', low_memory=False)
    print(f"FINESS columns: {list(finess_df.columns[:5])}")

    finess_rows = sum(1 for _ in open(FINESS_PATH, encoding='latin-1', errors='replace')) - 1
    print(f"FINESS rows: {finess_rows:,}")
    assert finess_rows > 50_000, f"FINESS too few rows: {finess_rows:,} (expected >50K)"
    print("FINESS: OK")
except Exception as e:
    print(f"FINESS ERROR: {e}")
    import traceback; traceback.print_exc()


## 4. MSP — Maisons de Santé Pluriprofessionnelles

In [ ]:
MSP_PATH = f"{RAW}/msp.csv"
MSP_URL = "https://www.data.gouv.fr/fr/datasets/r/4d20c2f7-a8e4-4b46-b672-60caeac4b0c3"
MSP_MIN_SIZE = 50_000  # 50 KB minimum

try:
    if os.path.exists(MSP_PATH) and os.path.getsize(MSP_PATH) > MSP_MIN_SIZE:
        print(f"MSP: Already downloaded ({os.path.getsize(MSP_PATH):,} bytes)")
    else:
        print("Downloading MSP...")
        urllib.request.urlretrieve(MSP_URL, MSP_PATH)
        print(f"MSP: Downloaded ({os.path.getsize(MSP_PATH):,} bytes)")

    # Check if it's actually a CSV (not HTML redirect)
    with open(MSP_PATH, 'rb') as f:
        header = f.read(100)
    if b'<html' in header.lower() or b'<!doctype' in header.lower():
        raise ValueError("Got HTML instead of CSV - URL may be incorrect")

    msp_df = pd.read_csv(MSP_PATH, nrows=5, sep=None, engine='python', encoding='utf-8')
    print(f"MSP columns: {list(msp_df.columns[:5])}")

    msp_rows = sum(1 for _ in open(MSP_PATH, encoding='utf-8', errors='replace')) - 1
    print(f"MSP rows: {msp_rows:,}")
    assert msp_rows > 1_000, f"MSP too few rows: {msp_rows:,} (expected >1K)"
    print("MSP: OK")
except Exception as e:
    print(f"MSP ERROR: {e}")
    import traceback; traceback.print_exc()


## 5. INSEE Population par Commune

In [ ]:
INSEE_POP_PATH = f"{RAW}/insee_pop.csv"
# INSEE population légale 2021 - fichier communes
INSEE_POP_URL = "https://www.insee.fr/fr/statistiques/fichier/8680726/Communes_2023.csv"
INSEE_POP_MIN_SIZE = 500_000  # 500 KB minimum

try:
    if os.path.exists(INSEE_POP_PATH) and os.path.getsize(INSEE_POP_PATH) > INSEE_POP_MIN_SIZE:
        print(f"INSEE Pop: Already downloaded ({os.path.getsize(INSEE_POP_PATH):,} bytes)")
    else:
        print("Downloading INSEE Population...")
        req = urllib.request.Request(INSEE_POP_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            data = response.read()
        with open(INSEE_POP_PATH, 'wb') as f:
            f.write(data)
        print(f"INSEE Pop: Downloaded ({os.path.getsize(INSEE_POP_PATH):,} bytes)")

    # Validate
    try:
        pop_df = pd.read_csv(INSEE_POP_PATH, nrows=5, sep=';', encoding='utf-8')
    except Exception:
        pop_df = pd.read_csv(INSEE_POP_PATH, nrows=5, sep=';', encoding='latin-1')
    print(f"INSEE Pop columns: {list(pop_df.columns[:8])}")

    pop_rows = sum(1 for _ in open(INSEE_POP_PATH, encoding='latin-1', errors='replace')) - 1
    print(f"INSEE Pop rows: {pop_rows:,}")
    assert pop_rows > 34_000, f"INSEE Pop too few rows: {pop_rows:,} (expected >34K communes)"
    cols_lower = [c.lower() for c in pop_df.columns]
    assert any("depcom" in c or "com" == c or "codgeo" in c for c in cols_lower), f"INSEE Pop missing commune col: {list(pop_df.columns)}"
    print("INSEE Pop: OK")
except Exception as e:
    print(f"INSEE Pop ERROR: {e}")
    import traceback; traceback.print_exc()


## 6. FiLoSoFi — Revenus et Pauvreté par Commune

In [ ]:
FILOSOFI_PATH = f"{RAW}/filosofi.csv"
# FiLoSoFi 2021 - niveau commune
FILOSOFI_URL = "https://www.data.gouv.fr/fr/datasets/r/dc2f012b-11ec-426a-be89-3b7efc6eada5"
FILOSOFI_MIN_SIZE = 2_000_000  # 2 MB minimum

try:
    if os.path.exists(FILOSOFI_PATH) and os.path.getsize(FILOSOFI_PATH) > FILOSOFI_MIN_SIZE:
        print(f"FiLoSoFi: Already downloaded ({os.path.getsize(FILOSOFI_PATH):,} bytes)")
    else:
        print("Downloading FiLoSoFi...")
        urllib.request.urlretrieve(FILOSOFI_URL, FILOSOFI_PATH)
        print(f"FiLoSoFi: Downloaded ({os.path.getsize(FILOSOFI_PATH):,} bytes)")

    # Check if zip
    with open(FILOSOFI_PATH, 'rb') as f:
        magic = f.read(4)
    if magic == b'PK\x03\x04':
        print("FiLoSoFi: Got ZIP, extracting CSV...")
        zip_path = FILOSOFI_PATH + ".zip"
        os.rename(FILOSOFI_PATH, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            csv_files = [n for n in zf.namelist() if n.endswith('.csv')]
            print(f"  ZIP contains: {csv_files}")
            if csv_files:
                with zf.open(csv_files[0]) as src, open(FILOSOFI_PATH, 'wb') as dst:
                    dst.write(src.read())
                print(f"  Extracted: {csv_files[0]}")

    try:
        fil_df = pd.read_csv(FILOSOFI_PATH, nrows=5, sep=';', encoding='utf-8')
    except Exception:
        fil_df = pd.read_csv(FILOSOFI_PATH, nrows=5, sep=';', encoding='latin-1')
    print(f"FiLoSoFi columns: {list(fil_df.columns[:8])}")

    fil_rows = sum(1 for _ in open(FILOSOFI_PATH, encoding='latin-1', errors='replace')) - 1
    print(f"FiLoSoFi rows: {fil_rows:,}")
    assert fil_rows > 30_000, f"FiLoSoFi too few rows: {fil_rows:,} (expected >30K)"
    cols_lower = [c.lower() for c in fil_df.columns]
    assert any("tp60" in c or "tpmp" in c or "pauvrete" in c or "pauv" in c for c in cols_lower), f"FiLoSoFi missing poverty col: {list(fil_df.columns)}"
    print("FiLoSoFi: OK")
except Exception as e:
    print(f"FiLoSoFi ERROR: {e}")
    import traceback; traceback.print_exc()


## 7. Pathologies — Data Ameli

In [ ]:
PATHO_PATH = f"{RAW}/pathologies.csv"
# Data Ameli - effectifs par pathologie et territoire
PATHO_URL = "https://data.ameli.fr/api/explore/v2.1/catalog/datasets/effectifs/exports/csv?lang=fr&timezone=Europe%2FParis&use_labels=true&delimiter=%3B"
PATHO_MIN_SIZE = 1_000  # 1 KB minimum (department-level, small file)

try:
    if os.path.exists(PATHO_PATH) and os.path.getsize(PATHO_PATH) > PATHO_MIN_SIZE:
        print(f"Pathologies: Already downloaded ({os.path.getsize(PATHO_PATH):,} bytes)")
    else:
        print("Downloading Pathologies (data.ameli)...")
        req = urllib.request.Request(PATHO_URL, headers={
            'User-Agent': 'Mozilla/5.0',
            'Accept': 'text/csv,application/csv,text/plain,*/*'
        })
        with urllib.request.urlopen(req, timeout=60) as response:
            data = response.read()
        with open(PATHO_PATH, 'wb') as f:
            f.write(data)
        print(f"Pathologies: Downloaded ({os.path.getsize(PATHO_PATH):,} bytes)")

    patho_df = pd.read_csv(PATHO_PATH, nrows=5, sep=';', encoding='utf-8')
    print(f"Pathologies columns: {list(patho_df.columns[:5])}")

    patho_rows = sum(1 for _ in open(PATHO_PATH, encoding='utf-8', errors='replace')) - 1
    print(f"Pathologies rows: {patho_rows:,}")
    assert patho_rows > 50, f"Pathologies too few rows: {patho_rows:,} (expected >50)"
    print("Pathologies: OK")
except Exception as e:
    print(f"Pathologies ERROR: {e}")
    import traceback; traceback.print_exc()


## 8. Admin Express — GeoJSON Communes (IGN)

In [ ]:
GEO_PATH = f"{RAW}/communes_geo.geojson"
# Admin Express COG 2024 - communes GeoJSON (simplifié)
# Direct download from data.gouv.fr Admin Express resource
GEO_URL = "https://www.data.gouv.fr/fr/datasets/r/0e117c06-248f-45e5-8945-0e79d9136165"
GEO_MIN_SIZE = 1_000_000  # 1 MB minimum

try:
    if os.path.exists(GEO_PATH) and os.path.getsize(GEO_PATH) > GEO_MIN_SIZE:
        print(f"Admin Express: Already downloaded ({os.path.getsize(GEO_PATH):,} bytes)")
    else:
        print("Downloading Admin Express GeoJSON (may be large)...")
        def reporthook(count, block_size, total_size):
            if total_size > 0 and count % 200 == 0:
                pct = min(count * block_size / total_size * 100, 100)
                print(f"  {pct:.1f}%", end="\r")
        urllib.request.urlretrieve(GEO_URL, GEO_PATH, reporthook=reporthook)
        print(f"\nAdmin Express: Downloaded ({os.path.getsize(GEO_PATH):,} bytes)")

    geo_size = os.path.getsize(GEO_PATH)
    print(f"Admin Express: {geo_size:,} bytes")
    assert geo_size > GEO_MIN_SIZE, f"Admin Express file too small: {geo_size:,} bytes"

    # Quick JSON validity check
    with open(GEO_PATH, 'r', encoding='utf-8', errors='replace') as f:
        first_bytes = f.read(100)
    assert '{' in first_bytes or '[' in first_bytes, f"Not valid JSON/GeoJSON: {first_bytes[:50]}"
    print("Admin Express: OK")
except Exception as e:
    print(f"Admin Express ERROR: {e}")
    import traceback; traceback.print_exc()


## 9. Diagnostic Accès Soins Urgents

In [ ]:
URGENCES_PATH = f"{RAW}/urgences.xlsx"
# Diagnostic d'accès aux soins urgents - fichier commune
URGENCES_URL = "https://www.data.gouv.fr/fr/datasets/r/45cf9e04-f08b-43c5-9c17-0398f33d32f6"
URGENCES_MIN_SIZE = 500_000  # 500 KB minimum

try:
    if os.path.exists(URGENCES_PATH) and os.path.getsize(URGENCES_PATH) > URGENCES_MIN_SIZE:
        print(f"Urgences: Already downloaded ({os.path.getsize(URGENCES_PATH):,} bytes)")
    else:
        print("Downloading Urgences XLSX...")
        urllib.request.urlretrieve(URGENCES_URL, URGENCES_PATH)
        print(f"Urgences: Downloaded ({os.path.getsize(URGENCES_PATH):,} bytes)")

    # Read with pandas
    urg_df = pd.read_excel(URGENCES_PATH, nrows=5)
    print(f"Urgences columns: {list(urg_df.columns[:8])}")

    urg_all = pd.read_excel(URGENCES_PATH)
    urg_rows = len(urg_all)
    print(f"Urgences rows: {urg_rows:,}")
    assert urg_rows > 30_000, f"Urgences too few rows: {urg_rows:,} (expected >30K)"
    cols_lower = [str(c).lower() for c in urg_all.columns]
    assert any("temps" in c or "min" in c or "acces" in c or "tps" in c for c in cols_lower), f"Urgences missing temps/minutes col: {list(urg_all.columns)}"
    print("Urgences: OK")
except Exception as e:
    print(f"Urgences ERROR: {e}")
    import traceback; traceback.print_exc()


## 10. La Poste — Table Codes Postaux → Communes INSEE

In [ ]:
LAPOSTE_PATH = f"{RAW}/la_poste_cp_commune.csv"
# Base officielle des codes postaux
LAPOSTE_URL = "https://www.data.gouv.fr/fr/datasets/r/554590ab-ae62-40ac-8353-ee75162c05ee"
LAPOSTE_MIN_SIZE = 500_000  # 500 KB minimum

try:
    if os.path.exists(LAPOSTE_PATH) and os.path.getsize(LAPOSTE_PATH) > LAPOSTE_MIN_SIZE:
        print(f"La Poste: Already downloaded ({os.path.getsize(LAPOSTE_PATH):,} bytes)")
    else:
        print("Downloading La Poste CP→Commune table...")
        urllib.request.urlretrieve(LAPOSTE_URL, LAPOSTE_PATH)
        print(f"La Poste: Downloaded ({os.path.getsize(LAPOSTE_PATH):,} bytes)")

    # Check if HTML was returned instead of CSV
    with open(LAPOSTE_PATH, 'rb') as f:
        header_bytes = f.read(200)
    if b'<html' in header_bytes.lower():
        raise ValueError("Got HTML redirect instead of CSV")

    lp_df = pd.read_csv(LAPOSTE_PATH, nrows=5, sep=None, engine='python', encoding='utf-8')
    print(f"La Poste columns: {list(lp_df.columns)}")

    lp_rows = sum(1 for _ in open(LAPOSTE_PATH, encoding='utf-8', errors='replace')) - 1
    print(f"La Poste rows: {lp_rows:,}")
    assert lp_rows > 30_000, f"La Poste too few rows: {lp_rows:,} (expected >30K)"
    cols_lower = [c.lower() for c in lp_df.columns]
    assert any("code_postal" in c or "postal" in c or "cp" == c for c in cols_lower), f"La Poste missing postal code col: {list(lp_df.columns)}"
    assert any("commune" in c or "insee" in c or "code_commune" in c for c in cols_lower), f"La Poste missing commune col: {list(lp_df.columns)}"
    print("La Poste: OK")
except Exception as e:
    print(f"La Poste ERROR: {e}")
    import traceback; traceback.print_exc()


## Summary — Status de tous les fichiers

In [ ]:
import os

datasets = [
    ("APL",           f"{RAW}/apl.csv",                    30_000,    1_000_000),
    ("RPPS",          f"{RAW}/rpps.csv",                 1_000_000,  100_000_000),
    ("FINESS",        f"{RAW}/finess.csv",                  50_000,    5_000_000),
    ("MSP",           f"{RAW}/msp.csv",                      1_000,       50_000),
    ("INSEE Pop",     f"{RAW}/insee_pop.csv",               34_000,      500_000),
    ("FiLoSoFi",      f"{RAW}/filosofi.csv",                30_000,    2_000_000),
    ("Pathologies",   f"{RAW}/pathologies.csv",                 50,        1_000),
    ("Admin Express", f"{RAW}/communes_geo.geojson",           None,    1_000_000),
    ("Urgences",      f"{RAW}/urgences.xlsx",                30_000,      500_000),
    ("La Poste",      f"{RAW}/la_poste_cp_commune.csv",    30_000,      500_000),
]

print(f"{'Dataset':<20} {'File':<35} {'Size (MB)':>10} {'Status'}")
print("-" * 85)
all_ok = True
for name, path, min_rows, min_size in datasets:
    if os.path.exists(path):
        size = os.path.getsize(path)
        size_mb = size / 1_000_000
        if size < min_size:
            status = f"WARN: small ({size:,} bytes)"
            all_ok = False
        else:
            status = "OK"
        print(f"{name:<20} {os.path.basename(path):<35} {size_mb:>10.2f}  {status}")
    else:
        print(f"{name:<20} {os.path.basename(path):<35} {'N/A':>10}  MISSING")
        all_ok = False

print()
if all_ok:
    print("ALL DATASETS PRESENT AND WITHIN SIZE BOUNDS")
else:
    print("WARNING: Some datasets missing or too small — check errors above")

# Final assertions
missing = [name for name, path, _, _ in datasets if not os.path.exists(path)]
assert len(missing) == 0, f"Missing files: {missing}"
print(f"\nTotal files in {RAW}/: {len(os.listdir(RAW))}")
